In [ ]:
import os
import numpy as np
import pandas as pd
import time
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import models, QdrantClient
import cohere


cohere_api_key = os.getenv('COHERE_API_KEY')
co = cohere.ClientV2(cohere_api_key)

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

encoder = SentenceTransformer('all-MiniLM-L6-v2')
reranker = CrossEncoder('ms-marco-MiniLM-L6-v2')
qdrant = QdrantClient(":memory:")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [7]:
dataset = load_dataset("explodinggradients/fiqa", "corpus", split="corpus[:1000]")
print("Dataset size:", len(dataset))
print(dataset[0])

data/corpus/corpus.parquet: reconstructing file:   0%|          |  0.00B / 27.7MB            

data/corpus/corpus.parquet: downloading bytes:           |  0.00B            

Generating corpus split: 0 examples [00:00, ? examples/s]

Dataset size: 1000
{'doc': "I'm not saying I don't like the idea of on-the-job training too, but you can't expect the company to do that. Training workers is not their job - they're building software. Perhaps educational systems in the U.S. (or their students) should worry a little about getting marketable skills in exchange for their massive investment in education, rather than getting out with thousands in student debt and then complaining that they aren't qualified to do anything."}


### We will be using 'corpus' config for code dev but using a sample of 1k docs.
### 'ragas_eval_v3' & 'main' config for experimentation and testing and deployment. It contains 5.5k train samples, 500 val and 648 test samples
### 'corpus' for final deployment. It contains all 57,638 final documents but no Q&A samples for testing or development

In [18]:
doc_objs = []
for i in range(len(dataset)):
    doc_obj = {
        "doc_id": i,
        "doc_text": " ".join(dataset[i]["doc"].split()),
        "word_len": len(dataset[i]["doc"].split()),
        "char_len": len(dataset[i]["doc"])
    }
    doc_objs.append(doc_obj)

In [ ]:
len_lst=[]
for i in range(len(doc_objs)):
    len_lst.append(doc_objs[i]["word_len"])
print(pd.Series(len_lst).describe())

count    1000.00000
mean      131.56700
std       128.47397
min         0.00000
25%        57.00000
50%        89.00000
75%       159.50000
max      1277.00000
dtype: float64


In [23]:
def chunking_fixed_size_with_overlap(text, chunk_size=100, overlap_fraction=0.2):

    text_words = text.split()
    overlap_int = int(chunk_size * overlap_fraction)
    
    chunks = []
    
    for i in range(0, len(text_words), chunk_size):
        chunk_words = text_words[max(i - overlap_int, 0): i + chunk_size]
        chunk = " ".join(chunk_words)
        chunks.append(chunk)
    
    return chunks

In [27]:
chunk_objs = []
for i in range(len(doc_objs)):
    text = doc_objs[i]["doc_text"]

    chunks = chunking_fixed_size_with_overlap(text)

    doc_chunk_objs = []
    for j,c in enumerate(chunks):
        chunk_obj = {
            "doc_id": i,
            "chunk_id": j,
            "chunk_text": c,
            "word_len": len(c.split()),
            "char_len": len(c)
        }
        doc_chunk_objs.append(chunk_obj)
    chunk_objs.extend(doc_chunk_objs)

    

In [32]:
print(chunk_objs[0])
print(doc_objs[0])

{'doc_id': 0, 'chunk_id': 0, 'chunk_text': "I'm not saying I don't like the idea of on-the-job training too, but you can't expect the company to do that. Training workers is not their job - they're building software. Perhaps educational systems in the U.S. (or their students) should worry a little about getting marketable skills in exchange for their massive investment in education, rather than getting out with thousands in student debt and then complaining that they aren't qualified to do anything.", 'word_len': 75, 'char_len': 460}
{'doc_id': 0, 'doc_text': "I'm not saying I don't like the idea of on-the-job training too, but you can't expect the company to do that. Training workers is not their job - they're building software. Perhaps educational systems in the U.S. (or their students) should worry a little about getting marketable skills in exchange for their massive investment in education, rather than getting out with thousands in student debt and then complaining that they aren'

In [ ]:
def embed_texts(texts, method, batch_size=96):
    if method=="cohere":
        embeddings=[]
        for i in range(0, len(texts), batch_size): # Running in batches bcoz Cohere trial API starts hittinh rate limit of 100k tokens/min
            batch = texts[i:i+batch_size]
            embeds = co.embed(
                texts=batch,
                input_type="search_document",
                model="embed-v4.0",
                embedding_types=["float"]
                ).embeddings.float # Since .embeddings return a .float list (EmbedTypeResponseEmbeddings object) and not a raw list of vectors
            embeddings.extend(embeds) # extend and not append since we are running in batches. Otherwise it will store each 96 embeddings in another sub-list indexed by batch_no
            time.sleep(60)
    elif method=="sentence_transformer":
        embeddings = encoder.encode(texts)
    else:
        raise ValueError(f"Invalid method: {method}")
    embeddings = np.array(embeddings)
    return embeddings

In [35]:
for i in range(len(chunk_objs)):
    chunk_embed = embed_texts(chunk_objs[i]["chunk_text"], method='sentence_transformer')
    chunk_objs[i]["chunk_embed"] = chunk_embed
print(chunk_embed.shape)
print(chunk_objs[0])

(384,)
{'doc_id': 0, 'chunk_id': 0, 'chunk_text': "I'm not saying I don't like the idea of on-the-job training too, but you can't expect the company to do that. Training workers is not their job - they're building software. Perhaps educational systems in the U.S. (or their students) should worry a little about getting marketable skills in exchange for their massive investment in education, rather than getting out with thousands in student debt and then complaining that they aren't qualified to do anything.", 'word_len': 75, 'char_len': 460, 'chunk_embed': array([-4.20321114e-02, -1.06007354e-02,  6.02464098e-03,  6.98938817e-02,
        5.50370291e-02,  3.66073735e-02,  7.24083092e-03, -1.61759350e-02,
       -6.11094534e-02, -4.57498357e-02,  2.52987873e-02,  1.07133955e-01,
       -1.95712857e-02,  1.81167561e-03, -6.75575584e-02, -4.20519747e-02,
        5.21229729e-02, -6.40632436e-02,  7.42716044e-02, -4.94503267e-02,
       -2.64491476e-02,  7.02336710e-03,  6.28138483e-02,  6.91

In [44]:
embeddings_size = chunk_objs[0]['chunk_embed'].shape[0]
print(embeddings_size)

384


In [45]:
def create_vector_db(embeddings_size, collection='fiqa_sample'):
    qdrant.create_collection(
        collection_name=collection,
        vectors_config=models.VectorParams(
            size=embeddings_size,
            distance=models.Distance.COSINE
        )
    )
    return

def add_data_to_vectordb(collection='fiqa_sample',chunk_objs=chunk_objs):
    qdrant.upload_points(
        collection_name=collection,
        points=[
            models.PointStruct(
                id=idx,
                vector=chunk_objs[idx]['chunk_embed'].tolist(),
                payload={
                    "doc_id": chunk_objs[idx]['doc_id'],
                    "chunk_id": chunk_objs[idx]['chunk_id'],
                    "chunk_text": chunk_objs[idx]['chunk_text'],
                    "word_len": chunk_objs[idx]['word_len'],
                    "char_len": chunk_objs[idx]['char_len']
                },
            ) for idx in range(len(chunk_objs))
        ]
    )
    return

In [46]:
create_vector_db(embeddings_size, collection='fiqa_sample')
add_data_to_vectordb(collection='fiqa_sample',chunk_objs=chunk_objs)

In [61]:
def search(query, method='sentence_transformer', collection='fiqa_sample', num_results=3):
    if method=="cohere":
        query_embed = co.embed(
            texts=[query],
            input_type="search_query",
            model="embed-v4.0",
            embedding_types=['float']
            ).embeddings.float[0] # .embeddings.float returns a list of vectors
    elif method=="sentence_transformer":
        query_embed = encoder.encode(query).tolist()

    hits = qdrant.query_points(
        collection_name = collection,
        query = query_embed,
        limit=num_results,
        with_vectors=True,
        with_payload=True,
    )
    search_results = []
    for hit in hits.points:
        result = {
            "chunk_id": hit.payload['chunk_id'], 
            "doc_id": hit.payload['doc_id'],
            "chunk_text": hit.payload['chunk_text'],
            "score": hit.score}
        search_results.append(result)
    
    return search_results

In [62]:
query =  'How to deposit a cheque issued to an associate in my business into my business account?'

search_results = search(query,collection='fiqa_sample')
for result in search_results:
    print(result)

{'chunk_id': 0, 'doc_id': 160, 'chunk_text': '"I expect the company wanted to pay you for a product (on a purchase order) rather than as a contract laborer. Whatever. Would they be willing to re-issue the check to you as a sole proprietor of a business named ABC Consulting (or anything like that)? You can register your sole proprietor business with the state using a ""Doing Business As"" (DBA, or fictitious name), and then open the bank account for your business using the check provided by the customer as the first deposit. (There is likely a smaller registration fee for the DBA.) If they won\'t re-issue the', 'score': 0.5874830036511175}
{'chunk_id': 1, 'doc_id': 160, 'chunk_text': "customer as the first deposit. (There is likely a smaller registration fee for the DBA.) If they won't re-issue the check and you have to go the LLC route... Scrounge up $125 doing odd jobs or borrowing from a friend or parents. Seriously, anyone can earn that amount of money in a week or two. Besides the 